In [ ]:
import functools as ft
import pathlib
import time
import typing

import jax
import jax.experimental.sparse as jax_sprs
import jax.numpy as jnp
import matplotlib.patheffects as patheffects
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvista as pv
import pyvistaqt as pvqt
import scipy.interpolate as spy_interp

import crops.src.bc as _bc
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.mma as _mma
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz
import crops.tools.geom as _geom

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.high_res_plot_settings)


In [ ]:
tgt_name = "k_008_right_4"
results_dir = pathlib.Path(f"results/paper/stent/{tgt_name}")
results_dir.mkdir(parents=True, exist_ok=True)

### Material

In [ ]:
shear_modulus_gel, shear_modulus_elastomer = 1.0e-3, 5.0e-2  # kN/mm^2
chi_gel, chi_elastomer = 0.2, 2.5

### Solvent

In [ ]:
solvent = _fe.FlorySolvent(
  bath_temperature=310.15,
  molar_volume=1.8e4,  # mm^3/mol
  ref_chem_potential=0.0,
)

### Load stepping potential

In [ ]:
num_load_steps = 50

dry_chem_potential = -1.0e5
wet_chem_potential = -12.0
t = jnp.linspace(0.0, 1.0, num_load_steps + 1)[1:]
ramp = t**0.03
chem_potentials = dry_chem_potential + (wet_chem_potential - dry_chem_potential) * ramp

plt.plot(chem_potentials, "r*")
plt.xlabel("load step")
plt.ylabel("chem potential")

### Continuation parameter

In [ ]:
continuation_epochs = 60
thresh_beta_max = 128.0
grayness_start, grayness_min = 0.3, 0.02
beta_rate = (thresh_beta_max - 1.0) / continuation_epochs
grayness_rate = (grayness_start - grayness_min) / continuation_epochs

### Solver settings

In [ ]:
nonlinear_settings = {
  "max_iter": 100,
  "threshold": 1e-8,
  "abs_threshold": 1e-12,
  "lam_min": 0.01,
  "line_search_max_iter": 12,
  "line_search_shrink": 0.5,
  "line_search_alpha_min": 1e-6,
  "line_search_armijo_c": 1e-4,
}
linear_settings = {
  "solver": _solver.LinearSolvers.SCIPY_SPARSE,
  "rtol": 1e-8,
}
bisection_settings = {
  "lower": 1e-6,
  "upper": 1.0 - 1e-6,
  "tol": 1e-6,
  "max_iter": 200,
}

### Basleine

In [ ]:
# Every study changes one entry of this dictionary. The aspect ratio is the window's
# L/R from the carotid extraction script.
baseline_settings = {
  "target_file": f"crops/data/stent_centerlines/{tgt_name}.csv",
  "unit_cell_file": "crops/data/stent_designs/to.png",
  "aspect_ratio": 30.0,  # stent length to radius
  "wire_radius": 0.2,  # mm
  "num_elems": 300,  # along the length
  "num_fibers": 180,  # along the cicrum
  "num_cells_axial": 12,  # repetition of unit cell along length
  "num_cells_circum": 6,  # repetition of unit cell along circum
  "filter_radius": 0.5,  # mm
}

### Problem setup

In [ ]:
class StentSetup(typing.NamedTuple):
  """Stores every target-dependent and discretization-dependent object of one run.

  One instance is built per study case. Element and fiber counts, element lengths,
  fiber offsets, and fiber areas are read from `rod` rather than duplicated here.

  Attributes:
    rod: `Rod` of the straight dry stent of length `L` along `+x`.
    problem: `CosseratProblem` holding the rod, the clamp, the Newton solver, and
      the solvent.
    filter_matrix: Sparse matrix of shape `(num_elems, num_elems)` of the normalized
      linear density filter along the stent axis.
    strut_mask: Boolean array of shape `(num_elems, num_fibers)` that is `True` where
      the strut pattern leaves material. Void fibers carry design variables with no
      effect, so the constraints and the plots use this to ignore them.
    target_nodes: Array of shape `(num_nodes, 3)` [mm] of target positions sampled at
      the reference arc coordinate of every rod node.
    target_curvature: Array of shape `(num_elems, 3)` of the target Darboux vector in
      its Bishop frame multiplied by `L`, so every entry is dimensionless.
  """

  rod: _mesher.Rod
  problem: _fe.CosseratProblem
  filter_matrix: jax_sprs.BCOO
  strut_mask: np.ndarray
  target_nodes: jnp.ndarray
  target_curvature: jnp.ndarray


### Problem builder

In [ ]:
def build_setup(
  target_file: str,
  unit_cell_file: str,
  aspect_ratio: float,
  wire_radius: float,
  num_elems: int,
  num_fibers: int,
  num_cells_axial: int,
  num_cells_circum: int,
  filter_radius: float,
  curv_mitig_factor: float = 1.0,
) -> StentSetup:
  """Builds the stent rod, its strut pattern, the solver, the filter, and the target.

  The rod represents the stent as a whole, so its cross-section is the annular wall
  of the tube. The fibers are circumferential stations around that wall and the
  elements are axial stations along it, which makes the pair `(element, fiber)` the
  unrolled cylindrical surface and lets the strut pattern map onto it directly. The
  rod takes the length `L` of the target and the radius `R_s = L / aspect_ratio`, so
  the target curvature and the reachable curvature are compared at the stent's
  physical slenderness.

  Algorithm.
    1. Read the target centerline, convert it from m to mm, set `L` to its arc
       length, and set `R_s = L / aspect_ratio`.
    2. Place `num_fibers` fibers around a circle of radius `R_s`, and binarize and
       tile the unit cell into the strut pattern on the `(element, fiber)` grid.
    3. Give every fiber the area of one strut, `π r_w²`, multiplied by the pattern,
       since a perforation removes material rather than softening it. The struts are
       treated as points, so their own second moments vanish.
    4. Build a straight rod along `+x` with the seed binormal along `+z`, and clamp
       all six dofs of node 0 for all `num_steps` chemical potential increments. The
       rest of the stent is free in 3D.
    5. Assemble the Newton solver and the swelling problem.
    6. Build the length-wise density filter with a cutoff in physical length, so
       refining the mesh leaves the regularization unchanged.
    7. Sample the target at the rod nodes, compute its Bishop-frame Darboux vector
       per unit target arc length with the same seed binormal, and multiply by `L`.
       The target starts along `+x` like the dry stent, so both frames agree at the
       clamp and stay twist-free.

  Args:
    target_file: CSV of the target centerline in m with columns `s`, `x`, `y`, `z`,
      as written by the carotid extraction script.
    unit_cell_file: Image of one strut unit cell, with rows along the circumference
      and columns along the axis.
    aspect_ratio: Stent slenderness `L / R_s`, positive, typically 5 to 30 at
      clinical carotid sizes.
    wire_radius: Strut radius `r_w` [mm], positive and well below `R_s`. It scales
      stiffness only and leaves the free-swelling shape unchanged.
    num_elems: Number of axial stations, a multiple of `num_cells_axial`.
    num_fibers: Number of circumferential stations, a multiple of
      `num_cells_circum`.
    num_cells_axial: Number of unit cells along the stent length.
    num_cells_circum: Number of unit cells around the circumference.
    filter_radius: Cutoff `R` [mm] of the density filter along the axis. A value
      below one element length leaves the design unfiltered.
    curv_mitig_factor: Sometimes the target might be too curved. That involves
      messing around with materials etc. For simplicity and illustration, we might
      scale target curvature be a small factor.

  Returns:
    A `StentSetup` holding every object one optimization run needs.
  """
  length_scale = 1e3  # convert m to mm. NOTE: Assumes the rod csv data in m
  target = pd.read_csv(target_file, comment="#")
  target_arc = length_scale * target["s"].values
  target_curve_fn = spy_interp.CubicSpline(
    target_arc, length_scale * target[["x", "y", "z"]].values
  )

  rod_length = float(target_arc[-1])
  tube_radius = rod_length / aspect_ratio

  strut_pattern = _geom.create_stent_mask(
    unit_cell=plt.imread(unit_cell_file),
    num_elems=num_elems,
    num_fibers=num_fibers,
    num_cells_axial=num_cells_axial,
    num_cells_circum=num_cells_circum,
  )
  rod = _mesher.build_rod_geometry(
    analytical_curve=lambda arc: jnp.array([arc, 0.0, 0.0]),
    parametric_coords=jnp.linspace(0.0, rod_length, num_elems + 1),
    fiber_offsets=jnp.asarray(
      _geom.annular_fiber_offsets(
        num_elems=num_elems,
        num_fibers=num_fibers,
        semi_axis_d1=tube_radius,
      )
    ),
    fiber_areas=jnp.asarray(strut_pattern * np.pi * wire_radius**2),
    fiber_mass_density=jnp.asarray(strut_pattern * 1100.0),
    fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),
    seed_binormal=jnp.array([0.0, 0.0, 1.0]),
  )

  clamped_dofs = (
    _utils.Translation.X,
    _utils.Translation.Y,
    _utils.Translation.Z,
    _utils.Rotation.THETA_X,
    _utils.Rotation.THETA_Y,
    _utils.Rotation.THETA_Z,
  )
  clamp = [
    _bc.DirichletBC(
      values=(dof, [np.array([0.0]) for _ in range(num_load_steps)]),
      node_targets=[np.array([0]) for _ in range(num_load_steps)],
      name=f"clamp_{dof}",
    )
    for dof in clamped_dofs
  ]
  bc = _bc.process_boundary_conditions(rod=rod, boundary_conditions_list=clamp)

  problem = _fe.CosseratProblem(
    rod=rod,
    bc=bc,
    nr_solver=_solver.NewtonRaphson(
      solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
      nonlinear_settings=nonlinear_settings,
      linear_settings=linear_settings,
      verbose=True,
    ),
    solvent=solvent,
    bisection_settings=bisection_settings,
  )

  arc_coords = np.asarray(rod.arc_coords)
  filter_matrix = _utils.create_density_filter(
    coords=0.5 * (arc_coords[:-1] + arc_coords[1:])[:, None],
    cutoff_distance=filter_radius,
  )

  target_nodes = jnp.asarray(target_curve_fn(arc_coords))
  target_kinematics = _mesher.compute_curvature_from_centerline(
    node_coords=target_nodes,
    seed_binormal=jnp.array([0.0, 0.0, 1.0]),
  )
  return StentSetup(
    rod=rod,
    problem=problem,
    filter_matrix=filter_matrix,
    strut_mask=strut_pattern > 0.5,
    target_nodes=target_nodes,
    target_curvature=target_kinematics.curvature * rod_length * curv_mitig_factor,
  )

### Objective

In [ ]:
def compute_pseudodensity(
  design_vars: jnp.ndarray,
  thresh_beta: float,
  setup: StentSetup,
) -> jnp.ndarray:
  """Maps the flat design vector to the filtered and projected fiber pseudodensity.

  Each circumferential station is filtered independently along the stent axis by the
  normalized linear filter `W`, giving `ρ̃_{e,k} = Σ_j W_{ej} x_{j,k}`. The result is
  then projected toward zero or one by the smoothed Heaviside function
  `ρ̂ = [tanh(βη) + tanh(β(ρ̃ − η))] / [tanh(βη) + tanh(β(1 − η))]` with `η = 1/2`.
  A value of one is gel and a value of zero is elastomer. Void fibers carry
  pseudodensity too, but their area is floored, so it has no mechanical effect.

  Args:
    design_vars: Array of shape `(num_elems * num_fibers,)` of design variables in
      `[0, 1]`, ordered element by element.
    thresh_beta: Projection sharpness `β`, typically annealed from 1 to 64.
    setup: `StentSetup` supplying the rod and the filter.

  Returns:
    Array of shape `(num_elems, num_fibers)` of pseudodensity values in `[0, 1]`.
  """
  design = design_vars.reshape(setup.rod.fibers.num_elems, setup.rod.fibers.num_fibers)
  filtered = setup.filter_matrix @ design
  return _utils.threshold_filter(density=filtered, beta=thresh_beta)


def compute_objective(
  design_vars: jnp.ndarray,
  thresh_beta: float,
  setup: StentSetup,
) -> tuple[jnp.ndarray, jnp.ndarray, tuple[jnp.ndarray, jnp.ndarray]]:
  """Computes the log relative curvature mismatch, its gradient, and the solved state.

  The ratio inside the logarithm is one for any layout that stays straight, which
  includes the uniform initial design. The objective therefore starts at zero and
  decreases, and its exponential reads as the fraction of the straight-stent
  mismatch that remains. Both curvatures are multiplied by the stent length `L`, so
  the comparison is scale-free and the swelling stretch never enters it. All three
  Darboux components enter, so spurious twist is penalized as well as bending.

  Algorithm.
    1. Map the design vector to the pseudodensity `ρ̂` by filtering and projection.
    2. Interpolate `G = ρ̂ G_gel + (1 − ρ̂) G_el` and `χ = ρ̂ χ_gel + (1 − ρ̂) χ_el` at
       every fiber of every element.
    3. Solve the swelling equilibrium along the chemical potential ramp.
    4. Recover the deformed nodes and directors, and compute the Darboux vector `κ_e`
       of every element per unit reference arc length.
    5. Evaluate `J = log( Σ_e ‖κ_e L − κ̂_e L‖² / Σ_e ‖κ̂_e L‖² )`, where `κ̂_e` is the
       target curvature.
    6. Differentiate steps 1 to 5 in reverse mode with respect to the design vector.

  Args:
    design_vars: Array of shape `(num_elems * num_fibers,)` of design variables in
      `[0, 1]`.
    thresh_beta: Projection sharpness `β`, typically annealed from 1 to 64.
    setup: `StentSetup` supplying the rod, the problem, the filter, and the target.

  Returns:
    A tuple `(objective, grad_objective, (pseudodensity, u_star))`. `objective` has
      shape `(1,)` and equals zero for a straight stent. `grad_objective` has shape
      `(num_design_var, 1)`. `pseudodensity` has shape `(num_elems, num_fibers)`.
      `u_star` has shape `(num_dofs,)` [mm, rad] and holds the converged
      displacements and rotations.
  """
  rod_length = setup.rod.arc_coords[-1]
  reference = jnp.einsum("ec,ec->", setup.target_curvature, setup.target_curvature)

  def objective_fn(
    design_vars: jnp.ndarray,
  ) -> tuple[jnp.ndarray, tuple[jnp.ndarray, jnp.ndarray]]:
    """Evaluates `J` and returns the pseudodensity and solved state as auxiliary."""
    pseudodensity = compute_pseudodensity(
      design_vars=design_vars, thresh_beta=thresh_beta, setup=setup
    )
    shear_moduli = shear_modulus_gel * pseudodensity + shear_modulus_elastomer * (
      1.0 - pseudodensity
    )
    flory_parameters = chi_gel * pseudodensity + chi_elastomer * (1.0 - pseudodensity)

    u_star, _ = setup.problem.solve(
      u0=jnp.zeros(setup.rod.num_dofs),
      shear_moduli=shear_moduli,
      flory_parameters=flory_parameters,
      chem_potentials=chem_potentials,
    )

    coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(
      rod=setup.rod,
      u=u_star,
    )
    kinematics = _mesher.compute_discrete_cosserat_kinematics(
      node_coords=coords,
      directors=directors,
      arc_coords=setup.rod.arc_coords,
    )

    error = kinematics.curvature * rod_length - setup.target_curvature
    mismatch = jnp.einsum("ec,ec->", error, error)
    return mismatch / reference, (pseudodensity, u_star)

  (objective, aux), grad_objective = jax.value_and_grad(objective_fn, has_aux=True)(
    design_vars
  )
  return objective.reshape(1), grad_objective.reshape(-1, 1), aux


### Constraints

In [ ]:
def compute_constraints(
  design_vars: jnp.ndarray,
  thresh_beta: float,
  max_grayness: float,
  max_gel_fraction: float,
  setup: StentSetup,
) -> tuple[jnp.ndarray, jnp.ndarray]:
  """Computes the normalized grayness and gel fraction constraints and their gradients.

  Both measures are averaged over the fibers the strut pattern keeps. Void fibers
  carry design variables with no mechanical effect, so including them would let
  them pad the gel fraction or hide grayness. Every kept fiber has the same area and
  every element the same reference length, so the masked mean of the pseudodensity
  equals the gel volume fraction of the stent. Both constraints are written as
  `value / bound − 1 ≤ 0`, so a positive entry is a violation and both entries are
  of order one. A gel fraction bound of one makes the second constraint inactive.

  Algorithm.
    1. Form the weights `w_{e,k} = m_{e,k} / Σ m`, where `m` is the strut mask.
    2. Map the design vector to the pseudodensity `ρ̂` by filtering and projection.
    3. Evaluate the grayness constraint `Σ w ρ̂ (1 − ρ̂) / γ_max − 1`.
    4. Evaluate the gel fraction constraint `Σ w ρ̂ / V_max − 1`.
    5. Differentiate each constraint separately in reverse mode with respect to the
       design vector, and stack the values and gradients in the shapes MMA expects.

  Args:
    design_vars: Array of shape `(num_elems * num_fibers,)` of design variables in
      `[0, 1]`.
    thresh_beta: Projection sharpness `β`, typically annealed from 1 to 64.
    max_grayness: Permitted grayness `γ_max`, in `(0, 0.25]`, annealed toward a
      small final value.
    max_gel_fraction: Permitted gel volume fraction `V_max`, in `(0, 1]`.
    setup: `StentSetup` supplying the filter and the strut mask.

  Returns:
    A tuple `(constraints, grad_constraints)`. `constraints` has shape
      `(num_cons=2, 1)` and holds the grayness constraint followed by the gel
      fraction constraint. `grad_constraints` has shape `(num_cons=2, num_design_var)`.
  """
  weights = setup.strut_mask / np.sum(setup.strut_mask)

  def grayness_fn(design_vars: jnp.ndarray) -> jnp.ndarray:
    """Evaluates the normalized grayness constraint over the kept fibers."""
    pseudodensity = compute_pseudodensity(
      design_vars=design_vars,
      thresh_beta=thresh_beta,
      setup=setup,
    )
    grayness = jnp.einsum("ef,ef->", weights, pseudodensity * (1.0 - pseudodensity))
    return grayness / max_grayness - 1.0

  def gel_fraction_fn(design_vars: jnp.ndarray) -> jnp.ndarray:
    """Evaluates the normalized gel fraction constraint over the kept fibers."""
    pseudodensity = compute_pseudodensity(
      design_vars=design_vars,
      thresh_beta=thresh_beta,
      setup=setup,
    )
    return jnp.einsum("ef,ef->", weights, pseudodensity) / max_gel_fraction - 1.0

  grayness, grad_grayness = jax.value_and_grad(grayness_fn)(design_vars)
  gel_fraction, grad_gel_fraction = jax.value_and_grad(gel_fraction_fn)(design_vars)
  return (
    jnp.stack([grayness, gel_fraction]).reshape(-1, 1),
    jnp.stack([grad_grayness, grad_gel_fraction]),
  )

### Plotting codes

In [ ]:
def add_top_legend(
  ax: plt.Axes,
  handles: typing.Optional[list[plt.Artist]] = None,
) -> None:
  """Draws the legend as one framed row centered just above the axes.

  Args:
    ax: Axis that owns the legend.
    handles: Labeled artists to list, in order. The labeled artists of `ax` are used
      when `None`.
  """
  if handles is None:
    handles, _ = ax.get_legend_handles_labels()
  ax.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=len(handles),
    frameon=True,
    handlelength=1.8,
    columnspacing=1.2,
  )


def target_glow(color: str) -> list[patheffects.AbstractPathEffect]:
  """Returns path effects that draw a line on top of a wide, faint, solid band.

  The band keeps a dashed target line visible where a solid line of the same color
  lies on it. Legend entries copy path effects, so the legend shows the band too.

  Args:
    color: Color of the band, normally the line's own color.

  Returns:
    A list of path effects to pass as `path_effects` to `ax.plot`.
  """
  return [
    patheffects.Stroke(
      linewidth=6.0,
      foreground=color,
      alpha=0.25,
      dashes={"dash_offset": 0.0, "dash_list": None},
    ),
    patheffects.Normal(),
  ]


def set_title_above_legend(ax: plt.Axes, title: str) -> None:
  """Sets the axis title, lifted clear of a legend drawn above the axes.

  A legend inside the axes leaves the title at its usual pad.

  Args:
    ax: Axis to title.
    title: Title text.
  """
  pad = plt.rcParams["axes.titlepad"]
  legend = ax.get_legend()
  if legend is not None:
    overhang = legend.get_window_extent().y1 - ax.get_window_extent().y1
    pad += max(overhang, 0.0) * 72.0 / ax.figure.dpi
  ax.set_title(title, pad=pad)


def plot_centerlines_3d(
  setup: StentSetup,
  disp: np.ndarray,
) -> tuple[plt.Figure, plt.Axes]:
  """Plots the swollen stent centerline against the target centerline in 3D.

  The objective matches shape rather than size, so the target is scaled about the
  clamp by the mean stretch `ν̄ = Λ / L`, where `Λ` is the swollen length obtained
  from the node chords. Only lines are drawn, which matplotlib renders without the
  depth-ordering artifacts that surfaces suffer.

  Args:
    setup: `StentSetup` of the case the displacement belongs to.
    disp: Array of shape `(num_dofs,)` [mm, rad] of converged displacements and
      rotations.

  Returns:
    A tuple `(fig, ax)` holding the figure and its 3D axis.
  """
  coords, _, _ = _mesher.extract_coords_and_director_from_deformation(
    rod=setup.rod, u=jnp.asarray(disp)
  )
  coords = np.asarray(coords)
  mean_stretch = np.linalg.norm(np.diff(coords, axis=0), axis=-1).sum() / float(
    setup.rod.arc_coords[-1]
  )
  scaled_target = mean_stretch * np.asarray(setup.target_nodes)

  fig = plt.figure(figsize=(5.0, 5.0))
  ax = fig.add_subplot(projection="3d")
  ax.plot(
    scaled_target[:, 0],
    scaled_target[:, 1],
    scaled_target[:, 2],
    color="0.55",
    ls="--",
    lw=1.2,
    label=r"target",
  )
  ax.plot(
    coords[:, 0], coords[:, 1], coords[:, 2], color="k", lw=1.5, label="swollen stent"
  )
  ax.set_aspect("equal")
  ax.set_axis_off()
  ax.legend(frameon=False)
  return fig, ax


def build_stent_surface(
  setup: StentSetup,
  design: np.ndarray,
  disp: np.ndarray,
) -> tuple[pv.PolyData, pv.PolyData]:
  """Builds the swollen strut surface and the scaled target centerline as VTK meshes.

  Every kept fiber of every element becomes one quad on the swollen tube wall. The
  quad spans the fiber's sector of angular width `2π / N_f`, centered on the fiber's
  own angle, and runs between the element's two nodes. Each corner sits at
  `p = r_n + R_s (cos φ d₁,n + sin φ d₂,n)`, where `r_n` and `d₁,n`, `d₂,n` are the
  swollen node position and directors. The wall has zero thickness, which avoids the
  overlap a strut-area-based thickness produces on a thin tube. Corners shared by
  neighboring quads are merged, so a renderer can shade the tube smoothly across
  quads instead of facet by facet. The gel pseudodensity is stored as cell data
  named `gel`. The target is a polyline scaled about the clamp by the mean stretch
  `ν̄`, because the objective matches shape rather than size.

  Args:
    setup: `StentSetup` of the case the design belongs to.
    design: Array of shape `(num_elems, num_fibers)` of pseudodensity in `[0, 1]`.
    disp: Array of shape `(num_dofs,)` [mm, rad] of converged displacements and
      rotations.

  Returns:
    A tuple `(surface, target_line)`. `surface` is a `pv.PolyData` with one quad per
      kept fiber and element. `target_line` is a `pv.PolyData` polyline of the scaled
      target.
  """
  coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(
    rod=setup.rod, u=jnp.asarray(disp)
  )
  coords, directors = np.asarray(coords), np.asarray(directors)
  offsets = np.asarray(setup.rod.fibers.offsets)
  tube_radius = float(np.linalg.norm(offsets[0, 0]))
  sector_span = 2.0 * np.pi / setup.rod.fibers.num_fibers
  fiber_angles = np.arctan2(offsets[..., 1], offsets[..., 0])

  def wall_points(nodes: slice, angles: np.ndarray) -> np.ndarray:
    """Places points on the swollen tube wall at the given nodes and angles."""
    return coords[nodes][:, None] + tube_radius * (
      np.einsum("ef,ed->efd", np.cos(angles), directors[nodes, 0])
      + np.einsum("ef,ed->efd", np.sin(angles), directors[nodes, 1])
    )

  lower = fiber_angles - 0.5 * sector_span
  upper = fiber_angles + 0.5 * sector_span
  start_nodes, end_nodes = slice(None, -1), slice(1, None)
  corners = np.stack(
    [
      wall_points(nodes=start_nodes, angles=lower),
      wall_points(nodes=end_nodes, angles=lower),
      wall_points(nodes=end_nodes, angles=upper),
      wall_points(nodes=start_nodes, angles=upper),
    ],
    axis=2,
  )[setup.strut_mask]
  num_quads = corners.shape[0]
  faces = np.column_stack(
    [np.full(num_quads, 4), np.arange(4 * num_quads).reshape(num_quads, 4)]
  )
  surface = pv.PolyData(corners.reshape(-1, 3), faces.ravel())
  surface.cell_data["gel"] = np.asarray(design)[setup.strut_mask]
  surface = surface.clean(tolerance=1e-6)

  mean_stretch = np.linalg.norm(np.diff(coords, axis=0), axis=-1).sum() / float(
    setup.rod.arc_coords[-1]
  )
  target_line = pv.lines_from_points(mean_stretch * np.asarray(setup.target_nodes))
  return surface, target_line


def plot_stent_surface(
  setup: StentSetup,
  design: np.ndarray,
  disp: np.ndarray,
  plotter: typing.Optional[pv.Plotter] = None,
  window_size: tuple[int, int] = (1200, 900),
  show_target: bool = True,
) -> pv.Plotter:
  """Renders the swollen stent colored by its gel map, with the scaled target line.

  PyVista renders with a depth buffer and lighting, so the front and back of the
  tube are ordered correctly, unlike a matplotlib 3D axis. An existing plotter can
  be passed in, the way an axis is passed to the matplotlib functions. A
  `pvqt.BackgroundPlotter` gives a non-blocking interactive Qt window. Without one,
  a new `pv.Plotter` is created and returned unshown, so the caller can set the
  camera, choose a backend, or save a screenshot.

  The wall is smooth shaded and its edges are antialiased with 8x multisampling. The
  target is drawn as a tube of radius `0.06 R_s` rather than as a line, so it keeps
  its thickness relative to the stent in the enlarged images of `save_screenshot`.
  The background is white, which `save_screenshot` crops away.

  Args:
    setup: `StentSetup` of the case the design belongs to.
    design: Array of shape `(num_elems, num_fibers)` of pseudodensity in `[0, 1]`.
    disp: Array of shape `(num_dofs,)` [mm, rad] of converged displacements and
      rotations.
    plotter: Optional plotter to draw into, for example a `pvqt.BackgroundPlotter`.
      A new `pv.Plotter` is created when `None`.
    window_size: Render size `(width, height)` in pixels, used only when a new
      plotter is created.
    show_target: Whether to draw the scaled target centerline.

  Returns:
    The `pv.Plotter` holding the stent surface and, if requested, the target line.
  """
  surface, target_line = build_stent_surface(setup=setup, design=design, disp=disp)
  tube_radius = float(np.linalg.norm(np.asarray(setup.rod.fibers.offsets)[0, 0]))
  if plotter is None:
    plotter = pv.Plotter(window_size=window_size)
  plotter.add_mesh(
    surface,
    scalars="gel",
    cmap=_viz.elast_gel_cmap,
    clim=(0.0, 1.0),
    show_scalar_bar=False,
    smooth_shading=True,
  )
  if show_target:
    plotter.add_mesh(
      target_line.tube(radius=0.06 * tube_radius, n_sides=32),
      color="black",
      smooth_shading=True,
    )
  plotter.set_background("white")
  plotter.enable_anti_aliasing("msaa", multi_samples=8)
  return plotter


def save_screenshot(
  plotter: pv.Plotter,
  file_path: pathlib.Path,
  scale: int = 4,
) -> None:
  """Saves the plotter's current view at `scale` times its window size, cropped tight.

  VTK renders the enlarged image in tiles, so its size is not limited by the screen.
  This works on an interactive `pvqt.BackgroundPlotter` after the view has been set
  by hand. The white margin around the stent is cropped away, so the image drops
  into a figure without padding.

  Args:
    plotter: Plotter from `plot_stent_surface`, with its white background.
    file_path: Destination with a raster extension such as `.png`.
    scale: Resolution multiplier on the window size, positive.
  """
  image = plotter.screenshot(return_img=True, scale=scale)
  foreground = np.any(image[..., :3] < 255, axis=-1)
  rows = np.flatnonzero(foreground.any(axis=1))
  cols = np.flatnonzero(foreground.any(axis=0))
  plt.imsave(file_path, image[rows[0] : rows[-1] + 1, cols[0] : cols[-1] + 1])


def save_stent_views(
  setup: StentSetup,
  design: np.ndarray,
  disp: np.ndarray,
  file_stem: pathlib.Path,
  scale: int = 4,
  show_target: bool = True,
) -> None:
  """Saves the front, right, top, and isometric views of the stent as high-res PNGs.

  Each view uses parallel projection, so the front, right, and top views show true
  lengths in their planes. The front view looks along `+y` with `z` up, the right
  view along `−x` with `z` up, and the top view along `−z` with `y` up. The isometric
  view looks along `(−1, 1, −1)` with `z` up, so it sees the faces the other three
  views see. The dry stent lies along `+x` from the clamp. Rendering happens off
  screen, so the image size does not depend on the display.

  Algorithm.
    1. Draw the stent with `plot_stent_surface` into a 1500 x 1500 px off-screen
       plotter with parallel projection.
    2. For each view, form the screen axes `e_h`, `e_v` from the view direction `d`
       and the up vector made orthogonal to `d`, and project the stent and target
       points onto them.
    3. Center the camera on the projected bounding box, place it `2 ℓ` from the
       center along `d`, where `ℓ` is the scene diagonal, so nothing falls behind
       the near clipping plane, and set the parallel scale to half the longer side of
       the box plus a 2% margin.
    4. Save the view with `save_screenshot` as `<stem>_<view>.png`, whose longer side
       is about `1500 scale` px once the margin is cropped.

  Args:
    setup: `StentSetup` of the case the design belongs to.
    design: Array of shape `(num_elems, num_fibers)` of pseudodensity in `[0, 1]`.
    disp: Array of shape `(num_dofs,)` [mm, rad] of converged displacements and
      rotations.
    file_stem: Path without extension. The files `<stem>_front.png`,
      `<stem>_right.png`, `<stem>_top.png`, and `<stem>_iso.png` are written.
    scale: Resolution multiplier on the 1500 px window, positive. A value of 4 gives
      about 6000 px on the longer side.
    show_target: Whether to draw the scaled target centerline.
  """
  # Direction from the stent to the camera, and the screen-up direction, per view.
  views = {
    "front": ((0.0, -1.0, 0.0), (0.0, 0.0, 1.0)),
    "right": ((1.0, 0.0, 0.0), (0.0, 0.0, 1.0)),
    "top": ((0.0, 0.0, 1.0), (0.0, 1.0, 0.0)),
    "iso": ((1.0, -1.0, 1.0), (0.0, 0.0, 1.0)),
  }
  surface, target_line = build_stent_surface(setup=setup, design=design, disp=disp)
  points = np.vstack([surface.points, target_line.points])
  plotter = plot_stent_surface(
    setup=setup,
    design=design,
    disp=disp,
    plotter=pv.Plotter(off_screen=True, window_size=(1500, 1500)),
    show_target=show_target,
  )
  plotter.enable_parallel_projection()
  for view, (direction, up) in views.items():
    direction = np.asarray(direction) / np.linalg.norm(direction)
    up = np.asarray(up) - np.dot(up, direction) * direction
    up /= np.linalg.norm(up)
    screen_axes = np.stack([np.cross(up, direction), up], axis=1)
    projected = points @ screen_axes
    low, high = projected.min(axis=0), projected.max(axis=0)
    depth = points @ direction
    focal_point = (
      screen_axes @ (0.5 * (low + high)) + 0.5 * (depth.min() + depth.max()) * direction
    )
    plotter.camera.focal_point = focal_point
    plotter.camera.position = focal_point + 2.0 * plotter.length * direction
    plotter.camera.up = up
    plotter.camera.parallel_scale = 0.51 * np.max(high - low)
    plotter.reset_camera_clipping_range()
    save_screenshot(
      plotter=plotter, file_path=pathlib.Path(f"{file_stem}_{view}.png"), scale=scale
    )
  plotter.close()


def export_stent_surface(
  setup: StentSetup,
  design: np.ndarray,
  disp: np.ndarray,
  file_stem: pathlib.Path,
) -> None:
  """Writes the swollen strut surface and the scaled target line to VTK files.

  The meshes are those of `build_stent_surface`. The surface carries the gel
  pseudodensity as cell data named `gel`, for coloring in ParaView.

  Args:
    setup: `StentSetup` of the case the design belongs to.
    design: Array of shape `(num_elems, num_fibers)` of pseudodensity in `[0, 1]`.
    disp: Array of shape `(num_dofs,)` [mm, rad] of converged displacements and
      rotations.
    file_stem: Path without extension. The files `<stem>_stent.vtp` and
      `<stem>_target.vtp` are written.
  """
  surface, target_line = build_stent_surface(setup=setup, design=design, disp=disp)
  surface.save(f"{file_stem}_stent.vtp")
  target_line.save(f"{file_stem}_target.vtp")


def plot_unrolled_stent(
  setup: StentSetup,
  design: np.ndarray,
) -> tuple[plt.Figure, plt.Axes]:
  """Plots the gel map on the stent wall cut along its length and laid flat.

  The `(element, fiber)` grid is the unrolled cylindrical surface, so the design is
  drawn as an image whose horizontal axis is the axial position and whose vertical
  axis is the circumferential arc length `R_s θ`. Both axes are in mm at equal
  aspect, so the strip has the true proportions `L × 2π R_s` of the flattened wall.
  Void fibers are masked out, which leaves the strut pattern visible. No colorbar is
  drawn, since gel and elastomer take the two end colors of `elast_gel_cmap`.

  Args:
    setup: `StentSetup` of the case the design belongs to.
    design: Array of shape `(num_elems, num_fibers)` of pseudodensity in `[0, 1]`.

  Returns:
    A tuple `(fig, ax)` holding the figure and its axis.
  """
  tube_radius = float(np.linalg.norm(np.asarray(setup.rod.fibers.offsets)[0, 0]))
  rod_length = float(setup.rod.arc_coords[-1])
  layout = np.where(setup.strut_mask, np.asarray(design), np.nan)

  fig, ax = plt.subplots(figsize=(8.0, 3.0))
  ax.imshow(
    layout.T,
    origin="lower",
    interpolation="nearest",
    extent=(0.0, rod_length, 0.0, 2.0 * np.pi * tube_radius),
    cmap=_viz.elast_gel_cmap,
    vmin=0.0,
    vmax=1.0,
  )
  ax.set_aspect("equal")
  ax.set_xlabel("length (mm)")
  ax.set_ylabel("circum. (mm)")
  return fig, ax


def plot_curvature(
  setup: StentSetup,
  disp: np.ndarray,
) -> tuple[plt.Figure, plt.Axes]:
  """Plots the achieved against the target Darboux components along the stent.

  All three components are normalized by the stent length `L`, which is the form
  the objective compares. Solid lines are achieved and dashed lines are the target,
  with one color per component. Each target line sits on a faint band of its own
  color and is drawn under the achieved line, so the target stays visible where the
  two overlap. The twist `κ₃ L` should stay near zero, since isotropic swelling
  cannot produce twist. The legend runs in one row above the axes.

  Args:
    setup: `StentSetup` of the case the displacement belongs to.
    disp: Array of shape `(num_dofs,)` [mm, rad] of converged displacements and
      rotations.

  Returns:
    A tuple `(fig, ax)` holding the figure and its axis.
  """
  coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(
    rod=setup.rod, u=jnp.asarray(disp)
  )
  kinematics = _mesher.compute_discrete_cosserat_kinematics(
    node_coords=coords, directors=directors, arc_coords=setup.rod.arc_coords
  )
  arc_coords = np.asarray(setup.rod.arc_coords)
  rod_length = arc_coords[-1]
  elem_arc = 0.5 * (arc_coords[:-1] + arc_coords[1:]) / rod_length
  achieved = rod_length * np.asarray(kinematics.curvature)
  target = np.asarray(setup.target_curvature)
  clrs = ["olive", "navy", "red"]
  fig, ax = plt.subplots(figsize=(6.0, 3.5))
  for component, label in enumerate((r"$\kappa_1$", r"$\kappa_2$", r"$\kappa_3$")):
    ax.plot(
      elem_arc,
      target[:, component],
      color=clrs[component],
      ls="--",
      lw=1.0,
      path_effects=target_glow(color=clrs[component]),
    )
    ax.plot(
      elem_arc, achieved[:, component], color=clrs[component], lw=1.2, label=label
    )
  ax.plot(
    [],
    [],
    color="0.4",
    ls="--",
    lw=1.0,
    label="target",
    path_effects=target_glow(color="0.4"),
  )
  ax.set_xlim(0.0, 1.0)
  ax.set_xlabel(r"$s/L$")
  ax.set_ylabel(r"$\kappa L$")
  ax.spines[["top", "right"]].set_visible(False)
  add_top_legend(ax=ax)
  return fig, ax


def plot_convergence(
  history: dict[str, np.ndarray],
) -> tuple[plt.Figure, tuple[plt.Axes, plt.Axes]]:
  """Plots the objective and the two constrained measures against the epoch.

  The objective `ε_κ` is the relative curvature mismatch, equal to one for a straight
  stent, and sits on a logarithmic left axis. The masked grayness and gel fraction
  share a linear right axis. Every line has its own color and dash pattern, so the
  plot stays readable in grayscale. A dotted line marks the epoch at which the
  projection sharpness reaches `β_max`.

  Args:
    history: Dictionary returned by `run_optimization` or loaded from its `.npz`,
      providing `epoch`, `objective`, `grayness`, and `gel_fraction`, each of shape
      `(num_epochs,)`.

  Returns:
    A tuple `(fig, (ax_obj, ax_cons))` holding the figure, the objective axis, and
      the constraint axis.
  """
  fig, ax_obj = plt.subplots(figsize=(6.5, 3.5))
  ax_cons = ax_obj.twinx()
  (objective,) = ax_obj.semilogy(
    history["epoch"], history["objective"], color="k", ls="-", label="objective"
  )
  (grayness,) = ax_cons.plot(
    history["epoch"], history["grayness"], color="b", ls="--", label="grayness"
  )
  (gel_fraction,) = ax_cons.plot(
    history["epoch"], history["gel_fraction"], color="r", ls="-.", label="gel fraction"
  )
  ax_obj.axvline(continuation_epochs, color="0.6", ls=":", lw=0.8)
  ax_obj.set_xlabel("epoch")
  ax_obj.set_ylabel(r"objective $\varepsilon_\kappa$")
  ax_cons.set_ylabel("constraints")
  ax_cons.set_ylim(bottom=0.0)
  for ax in (ax_obj, ax_cons):
    ax.spines["top"].set_visible(False)
  add_top_legend(ax=ax_obj, handles=[objective, grayness, gel_fraction])
  return fig, (ax_obj, ax_cons)


def plot_stent_state(
  setup: StentSetup,
  design: np.ndarray,
  disp: np.ndarray,
  epoch: typing.Optional[int] = None,
  rel_error: typing.Optional[float] = None,
) -> dict[str, tuple[plt.Figure, plt.Axes]]:
  """Draws the centerline comparison, the unrolled gel map, and the curvature plot.

  Each view is its own figure, so any subset can be kept for the paper. The full 3D
  stent surface is rendered separately by `plot_stent_surface`, and written to disk
  by `export_stent_surface`.

  Args:
    setup: `StentSetup` of the case the design belongs to.
    design: Array of shape `(num_elems, num_fibers)` of pseudodensity in `[0, 1]`.
    disp: Array of shape `(num_dofs,)` [mm, rad] of converged displacements and
      rotations.
    epoch: Optional epoch shown in the title of every figure during monitoring.
    rel_error: Optional relative curvature error `ε_κ` added to the titles of the
      centerline and curvature figures. The unrolled gel map shows the epoch only.

  Returns:
    A dictionary mapping `centerlines`, `unrolled`, and `curvature` to their
      `(fig, ax)` pairs.
  """
  figures = {
    "centerlines": plot_centerlines_3d(setup=setup, disp=disp),
    "unrolled": plot_unrolled_stent(setup=setup, design=design),
    "curvature": plot_curvature(setup=setup, disp=disp),
  }
  if epoch is not None:
    for name, (_, ax) in figures.items():
      title = f"epoch {epoch}"
      if rel_error is not None and name != "unrolled":
        mantissa, exponent = f"{rel_error:.2e}".split("e")
        title += rf", $\varepsilon_\kappa = {mantissa} \times 10^{{{int(exponent)}}}$"
      set_title_above_legend(ax=ax, title=title)
  return figures

### Optimization

In [ ]:
def run_optimization(
  setup: StentSetup,
  max_gel_fraction: float,
  max_iter: int = 151,
  move_limit: float = 1e-3,
  save_interval: int = 3,
  monitor: bool = False,
) -> dict[str, np.ndarray]:
  """Runs MMA with synchronized projection and grayness continuation.

  The objective and constraints are bound to one setup and compiled once, so their
  inner functions and gradients are traced a single time per run. The history is
  returned as a dictionary of arrays that can be saved directly with `np.savez`.
  Converting every epoch's outputs to NumPy for MMA forces the device to finish, so
  the recorded wall time per epoch is exact. The first entry of `time_epoch` includes
  compilation, so the compile time can be estimated as
  `time_epoch[0] − median(time_epoch[1:])`.

  Algorithm.
    1. Bind the setup into the objective and the constraints and compile both.
    2. Start from the uniform design `x = 1/2`, which has zero natural curvature, so
       the objective starts at `log 1 = 0`.
    3. At epoch `k` set `β_k = min(β_max, 1 + k (β_max − 1) / K)` and
       `γ_max,k = max(γ_min, γ_0 − k (γ_0 − γ_min) / K)`, where `K` is
       `continuation_epochs`, so both reach their final values together.
    4. Evaluate the objective, the constraints, and their gradients, and update the
       design with one MMA step, timing the whole epoch.
    5. Record the objective, the masked grayness and gel fraction, and the epoch time
       every epoch. Every `save_interval` epochs and at the last epoch, also record
       the pseudodensity and the displacement, and draw the state if `monitor` is set.
    6. Stop when MMA reports convergence after `continuation_epochs`, or at
       `max_iter`.

  Args:
    setup: `StentSetup` of the case to optimize.
    max_gel_fraction: Permitted gel volume fraction `V_max` in `(0, 1]`. A value of
      one disables the constraint.
    max_iter: Maximum number of MMA epochs, larger than `continuation_epochs`.
    move_limit: MMA move limit per epoch, in `(0, 1]`.
    save_interval: Number of epochs between stored snapshots, positive.
    monitor: Whether to draw the stent state at every stored snapshot.

  Returns:
    A dictionary of arrays. `epoch`, `objective`, `grayness`, `gel_fraction`, and
      `time_epoch` have shape `(num_epochs,)`, with times in seconds. `saved_epoch`
      has shape `(num_saved,)`. `design` has shape
      `(num_saved, num_elems, num_fibers)`. `disp` has shape `(num_saved, num_dofs)`.
  """
  objective_and_grad = jax.jit(ft.partial(compute_objective, setup=setup))
  constraints_and_grad = jax.jit(ft.partial(compute_constraints, setup=setup))
  weights = setup.strut_mask / np.sum(setup.strut_mask)

  num_design_var = setup.rod.fibers.num_elems * setup.rod.fibers.num_fibers
  mma_params = _mma.MMAParams(
    max_iter=max_iter,
    kkt_tol=1e-6,
    step_tol=1e-6,
    move_limit=move_limit,
    num_design_var=num_design_var,
    num_cons=2,
    lower_bound=np.zeros((num_design_var, 1)),
    upper_bound=np.ones((num_design_var, 1)),
  )
  mma_state = _mma.init_mma(
    init_design_var=np.full((num_design_var, 1), 0.5),
    mma_params=mma_params,
  )
  keys = (
    "epoch",
    "objective",
    "grayness",
    "gel_fraction",
    "time_epoch",
    "saved_epoch",
    "design",
    "disp",
  )
  history = {key: [] for key in keys}

  while mma_state.epoch < max_iter and not (
    mma_state.is_converged and mma_state.epoch > continuation_epochs
  ):
    epoch = mma_state.epoch
    thresh_beta = min(thresh_beta_max, 1.0 + beta_rate * epoch)
    max_grayness = max(grayness_min, grayness_start - grayness_rate * epoch)
    design_vars = jnp.asarray(mma_state.x).reshape(-1)

    epoch_start = time.perf_counter()
    objective, grad_objective, (pseudodensity, u_star) = objective_and_grad(
      design_vars,
      thresh_beta=thresh_beta,
    )
    constraints, grad_constraints = constraints_and_grad(
      design_vars,
      thresh_beta=thresh_beta,
      max_grayness=max_grayness,
      max_gel_fraction=max_gel_fraction,
    )
    mma_state = _mma.update_mma(
      mma_state=mma_state,
      mma_params=mma_params,
      obj=np.asarray(objective, dtype=np.float64),
      grad_obj=np.asarray(grad_objective, dtype=np.float64),
      cons=np.asarray(constraints, dtype=np.float64),
      grad_cons=np.asarray(grad_constraints, dtype=np.float64),
    )
    time_epoch = time.perf_counter() - epoch_start

    layout = np.asarray(pseudodensity)
    rel_error = float(objective[0])
    history["epoch"].append(epoch)
    history["objective"].append(float(objective[0]))
    history["grayness"].append(float(np.sum(weights * layout * (1.0 - layout))))
    history["gel_fraction"].append(float(np.sum(weights * layout)))
    history["time_epoch"].append(time_epoch)

    if epoch % save_interval == 0:
      history["saved_epoch"].append(epoch)
      history["design"].append(layout)
      history["disp"].append(np.asarray(u_star))
      print(
        f"epoch {epoch:4d}  rel. error {rel_error:.3e}  "
        f"gray {history['grayness'][-1]:.3f}  gel {history['gel_fraction'][-1]:.3f}  "
        f"{time_epoch:.2f} s"
      )
      if monitor:
        plot_stent_state(
          setup=setup,
          design=layout,
          disp=np.asarray(u_star),
          epoch=epoch,
          rel_error=rel_error,
        )
        plt.show()
        plt.pause(1e-6)

  if history["saved_epoch"][-1] != epoch:
    history["saved_epoch"].append(epoch)
    history["design"].append(layout)
    history["disp"].append(np.asarray(u_star))
  return {key: np.asarray(value) for key, value in history.items()}

In [ ]:
baseline = build_setup(**baseline_settings)

### Baseline run - RUN CELL

In [ ]:
run = False

if run:
  

  history = run_optimization(setup=baseline, max_gel_fraction=1.0, monitor=True)

  np.savez(results_dir / "history.npz", **history, **baseline_settings)

### LOAD HISTORY

In [ ]:
history = np.load(results_dir / "history.npz")

In [ ]:
# The mean stretch tells you by how much the window's L/R overstates the dry L/R.
coords, _, _ = _mesher.extract_coords_and_director_from_deformation(
  rod=baseline.rod, u=jnp.asarray(history["disp"][-1])
)
mean_stretch = np.linalg.norm(
  np.diff(np.asarray(coords), axis=0), axis=-1
).sum() / float(baseline.rod.arc_coords[-1])
print(f"final relative error {np.exp(history['objective'][-1]):.3e}")
print(f"mean stretch {mean_stretch:.3f}")

figures = plot_stent_state(
  setup=baseline, design=history["design"][-1], disp=history["disp"][-1]
)
for name, (fig, _) in figures.items():
  fig.savefig(results_dir / f"{name}.pdf")
plt.show()

In [ ]:
fig, _ = plot_convergence(history=history)
fig.savefig(results_dir / "convergence.pdf")
plt.show()

# Tab separated, so the printed block pastes straight into a spreadsheet.
convergence = pd.DataFrame(
  {key: history[key] for key in ("epoch", "objective", "grayness", "gel_fraction")}
)
print(convergence.to_csv(sep="\t", index=False, float_format="%.6e"))

In [ ]:
final_design, final_disp = history["design"][-5], history["disp"][-5]
export_stent_surface(
  setup=baseline,
  design=final_design,
  disp=final_disp,
  file_stem=results_dir / "surface",
)

# Interactive Qt window, the equivalent of %matplotlib qt5.
plotter = plot_stent_surface(
  setup=baseline,
  design=final_design,
  disp=final_disp,
  plotter=pvqt.BackgroundPlotter(window_size=(1600, 1200)),
  show_target=False,
)

In [ ]:
# Set the view by hand in the Qt window first, then save it.
save_screenshot(plotter=plotter, file_path=results_dir / "stent_view.png", scale=4)

In [ ]:
save_stent_views(
  setup=baseline,
  design=final_design,
  disp=final_disp,
  file_stem=results_dir / "stent",
  show_target=False,
)